# Lekce 13 - Paměť agenta


## Nastavení

Tento zápisník ukazuje, jak vytvořit agenta pro rezervaci cestování s **perzistentní pamětí** pomocí **Microsoft Agent Framework** (MAF).

Naučíte se, jak různé typy paměti agenta — pracovní, krátkodobá a dlouhodobá — ovlivňují, jak agent uchovává a využívá informace v průběhu konverzací.

**Předpoklady:**
- Projekt Azure AI Foundry s nasazeným chatovacím modelem (např. `gpt-4o-mini`).
- Přihlášení pomocí Azure CLI — spusťte `az login` ve vašem terminálu.
- `AZURE_AI_PROJECT_ENDPOINT` — koncový bod vašeho projektu Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — název vašeho nasazeného modelu.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy paměti agenta

AI agenti mohou využívat různé druhy paměti, z nichž každý slouží jinému účelu:

### Pracovní paměť
Samotný tok konverzace — zprávy vyměněné během jedné relace. Agent se může odkazovat na dřívější zprávy ve stejném vlákně, aby udržel soudržnost. V MAF je to vytvořeno pomocí **`agent.create_session()`**, která vrací `AgentSession`.

### Krátkodobá paměť
Informace, které přetrvávají po dobu úkolu nebo relace, ale nejsou trvale uloženy. Například agent může během vícekrokové plánovací konverzace shromažďovat fakta a použít je k vytvoření konečného itineráře.

### Dlouhodobá paměť
Preference a fakta, která přetrvávají **napříč relacemi**. Vracející se uživatel by neměl muset opakovat své dietní omezení nebo styl cestování. Dlouhodobá paměť je obvykle podpořena externím úložištěm — databází, souborem nebo vektorovým indexem — a zpřístupněna agentovi prostřednictvím nástrojů.


## Pracovní paměť se sezeními

Nejjednodušší formou paměti je konverzační sezení. Když předáte stejné sezení (vytvořené pomocí `agent.create_session()`) do po sobě jdoucích volání `agent.run()`, agent vidí celou historii té konverzace a může si pamatovat dřívější detaily.

Vytvořme cestovního agenta a ukažme pracovní paměť v praxi.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent si správně vybavil rozpočet, protože obě zprávy sdílejí stejnou relaci. Toto je **pracovní paměť** — existuje pouze po dobu trvání relace.

### Co se stane s novým vláknem?

Pokud vytvoříme **novou** relaci, agent si nepamatuje předchozí konverzaci:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzor dlouhodobé paměti

Pro zapamatování uživatelských preferencí **napříč relacemi** potřebujeme perzistentní úložiště, které existuje mimo konverzační vlákno. Agent k tomuto úložišti přistupuje pomocí **nástrojů** — funkcí, které může volat pro ukládání a získávání informací.

Níže implementujeme jednoduché preferenční úložiště v paměti (v produkci byste ho podpojili k databázi nebo vektorovému indexu) a zpřístupníme ho jako nástroje, které může agent používat.

### Architektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scénář 1 — Uživatel poprvé rezervuje výlet k výročí

Sarah navštíví stránky poprvé. Agent by měl pomocí nástrojů uložit její preference a použít je k doporučení hotelů.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scénář 2 — Sarah se vrací týdny poté

Sarah začíná **zcela nové vlákno** (simulace nové relace). Pracovní paměť je prázdná, ale v dlouhodobém úložišti preferencí jsou stále uloženy její informace. Agent by je měl načíst a použít k personalizaci doporučení.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Souhrn

V této lekci jste se naučili tři typy paměti agenta a jak je implementovat pomocí Microsoft Agent Framework:

| Typ paměti | Mechanismus MAF | Doba platnosti |
|---|---|---|
| **Pracovní** | `agent.create_session()` | Jedna konverzace |
| **Krátkodobá** | Akumulovaný kontext v rámci vlákna | Jednotlivý úkol / relace |
| **Dlouhodobá** | Vnější úložiště přístupné přes `@tool` funkce | Mezi relacemi |

### Klíčová shrnutí
1. **`agent.create_session()`** poskytuje pracovní paměť — agent vidí kompletní historii konverzace v rámci relace.
2. **Nové relace ztrácejí kontext** — bez dlouhodobé paměti si agent nepamatuje minulé konverzace.
3. **`@tool` funkce** překonávají tento problém — umožňují agentovi ukládat a načítat informace z trvalého úložiště.
4. **Personalizace se časem zlepšuje** — čím více preferencí je uloženo, tím lepší jsou agentova doporučení.

### Praktické aplikace
- **Zákaznický servis**: Zapamatování historie a preferencí zákazníků
- **Osobní asistenti**: Udržení kontextu přes dny či týdny
- **Zdravotnictví**: Sledování informací a preferencí pacientů
- **E-commerce**: Personalizované nakupování podle historie

### Další kroky
- Nahradit slovník v paměti databází nebo vektorovým úložištěm (např. Azure AI Search)
- Přidat vypršení platnosti paměti pro časově citlivé informace
- Vytvářet multi-agentní systémy se sdílenou pamětí
- Prozkoumat Cognee notebook pro paměť podporovanou znalostními grafy


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí služby strojového překladu AI [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho původním jazyce by měl být považován za závazný zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nepřejímáme odpovědnost za jakákoliv nedorozumění nebo nesprávné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
